In [2]:
import os
import json
import glob
import numpy as np
import cv2
from skimage import measure
from tqdm import tqdm

def labelme_to_coco(labelme_folder, output_json):
    images, annotations = [], []
    ann_id = 1
    categories = [{"id": 1, "name": "object"}]

    for img_id, file in enumerate(tqdm(glob.glob(os.path.join(labelme_folder, "*.json")))):
        with open(file, 'r') as f:
            data = json.load(f)

        height = data["imageHeight"]
        width = data["imageWidth"]
        filename = data["imagePath"]

        images.append({
            "id": img_id,
            "file_name": filename,
            "height": height,
            "width": width
        })

        for shape in data["shapes"]:
            if shape["shape_type"] != "polygon":
                continue

            points = np.array(shape["points"], dtype=np.int32)
            mask = np.zeros((height, width), dtype=np.uint8)
            cv2.fillPoly(mask, [points], 1)

            segmentation = []
            contours = measure.find_contours(mask, 0.5)
            for contour in contours:
                contour = np.flip(contour, axis=1)
                segmentation.append(contour.ravel().tolist())

            x, y, w, h = cv2.boundingRect(points)

            annotations.append({
                "id": ann_id,
                "image_id": img_id,
                "category_id": 1,
                "segmentation": segmentation,
                "area": float(np.sum(mask)),
                "bbox": [x, y, w, h],
                "iscrowd": 0
            })
            ann_id += 1

    with open(output_json, "w") as f:
        json.dump({
            "images": images,
            "annotations": annotations,
            "categories": categories
        }, f)

labelme_to_coco("same-object-annotations", "same-object-coco.json")


100%|██████████| 3323/3323 [02:24<00:00, 23.06it/s]
